# data_canonical_snapshot_v1

目标：验证 `src.data` 可以构建 canonical snapshot，完成 qlib calendar 对齐、防未来函数校验、schema hash 和 sidecar 写出。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_canonical_snapshot, write_canonical_snapshot

CARD = REPO_ROOT / 'research/data_canonical_snapshot_v1'
SNAPSHOT_DIR = CARD / 'canonical_snapshot'
CARD.mkdir(parents=True, exist_ok=True)
{'card': str(CARD.relative_to(REPO_ROOT)), 'snapshot_dir': str(SNAPSHOT_DIR.relative_to(REPO_ROOT))}

{'card': 'research/data_canonical_snapshot_v1',
 'snapshot_dir': 'research/data_canonical_snapshot_v1/canonical_snapshot'}

## 构造原始数据与 qlib calendar

原始数据包含一个非交易日行。canonical snapshot 应只保留 calendar 内的行，并在 metadata 中记录剔除数量。

In [2]:
raw = pd.DataFrame([
    {'date': '2026-01-02', 'instrument': 'SOXL', 'close': 10.0, 'volume': 100},
    {'date': '2026-01-03', 'instrument': 'SOXL', 'close': 11.0, 'volume': 110},
    {'date': '2026-01-05', 'instrument': 'SOXL', 'close': 12.0, 'volume': 120},
])
calendar = pd.DatetimeIndex(pd.to_datetime(['2026-01-02', '2026-01-05']), name='date')
snapshot = build_canonical_snapshot(
    raw,
    name='unit_soxl_daily',
    universe=('SOXL',),
    calendar=calendar,
    calendar_path='data/qlib/soxl/calendars/day.txt',
    source_paths=('raw/unit.csv',),
    start='2026-01-01',
    end='2026-01-05',
    as_of='2026-01-05',
)
assert snapshot.frame['date'].dt.strftime('%Y-%m-%d').tolist() == ['2026-01-02', '2026-01-05']
assert snapshot.dataset.metadata['dropped_non_calendar_rows'] == 1
snapshot.dataset.to_dict()

{'calendar_path': 'data/qlib/soxl/calendars/day.txt',
 'data_paths': ['raw/unit.csv'],
 'end': '2026-01-05',
 'metadata': {'as_of': '2026-01-05',
  'calendar_row_count': 2,
  'canonical_frame_schema_version': 'data.canonical_frame.v1',
  'column_count': 4,
  'date_column': 'date',
  'dropped_non_calendar_rows': 1,
  'frame_schema_hash': 'd50304bef3df60b6ae09aa909f0226caf58f6f1b8a4d5b99e89f3074383613cb',
  'row_count': 2,
  'source_paths': ['raw/unit.csv']},
 'name': 'unit_soxl_daily',
 'schema_version': 'dataset.v1',
 'start': '2026-01-02',
 'universe': ['SOXL']}

## 防未来函数校验

任何晚于 `as_of` 的行应在 calendar 过滤前被拒绝，避免因为后续过滤掩盖数据越界。

In [3]:
future_raw = pd.DataFrame([
    {'date': '2026-01-02', 'instrument': 'SOXL', 'close': 10.0},
    {'date': '2026-01-06', 'instrument': 'SOXL', 'close': 12.0},
])
future_error = ''
try:
    build_canonical_snapshot(
        future_raw,
        name='unit_soxl_daily',
        universe=('SOXL',),
        calendar=pd.DatetimeIndex(pd.to_datetime(['2026-01-02', '2026-01-06']), name='date'),
        calendar_path='data/qlib/soxl/calendars/day.txt',
        source_paths=('raw/unit.csv',),
        start='2026-01-01',
        end='2026-01-06',
        as_of='2026-01-05',
    )
except ValueError as exc:
    future_error = str(exc)
assert 'future rows after as_of' in future_error
future_error

'future rows after as_of=2026-01-05: first=2026-01-06'

## 写出 snapshot sidecar

写出 `canonical_frame.csv` 和 `dataset_snapshot.json`，后续真实数据任务可以把输出目录换成 `data/...`。

In [4]:
export = write_canonical_snapshot(snapshot, SNAPSHOT_DIR)
saved_snapshot = json.loads((SNAPSHOT_DIR / 'dataset_snapshot.json').read_text(encoding='utf-8'))
saved_frame = pd.read_csv(SNAPSHOT_DIR / 'canonical_frame.csv')
assert export['dataset_schema_hash'] == snapshot.dataset.schema_hash()
assert saved_snapshot['dataset']['metadata']['frame_schema_hash'] == snapshot.dataset.metadata['frame_schema_hash']
assert saved_frame['date'].tolist() == ['2026-01-02', '2026-01-05']
export

{'schema_version': 'data.canonical_snapshot.export.v1',
 'dataset': {'calendar_path': 'data/qlib/soxl/calendars/day.txt',
  'data_paths': ['raw/unit.csv'],
  'end': '2026-01-05',
  'metadata': {'as_of': '2026-01-05',
   'calendar_row_count': 2,
   'canonical_frame_schema_version': 'data.canonical_frame.v1',
   'column_count': 4,
   'date_column': 'date',
   'dropped_non_calendar_rows': 1,
   'frame_schema_hash': 'd50304bef3df60b6ae09aa909f0226caf58f6f1b8a4d5b99e89f3074383613cb',
   'row_count': 2,
   'source_paths': ['raw/unit.csv']},
  'name': 'unit_soxl_daily',
  'schema_version': 'dataset.v1',
  'start': '2026-01-02',
  'universe': ['SOXL']},
 'dataset_schema_hash': 'b4b81ddd4b6bf9bfe93432e51202b2eb77abb08809e3906c46d02a9693d73b02',
 'paths': {'frame': 'canonical_frame.csv',
  'dataset_snapshot': 'dataset_snapshot.json'}}

In [5]:
summary = {
    'schema_version': 'research.data_canonical_snapshot_v1',
    'status': 'completed',
    'snapshot_dir': str(SNAPSHOT_DIR.relative_to(REPO_ROOT)),
    'row_count': snapshot.dataset.metadata['row_count'],
    'dropped_non_calendar_rows': snapshot.dataset.metadata['dropped_non_calendar_rows'],
    'frame_schema_hash': snapshot.dataset.metadata['frame_schema_hash'],
    'dataset_schema_hash': export['dataset_schema_hash'],
    'future_guardrail_error': future_error,
    'conclusion': 'canonical snapshot 已完成 qlib calendar 对齐、future row guardrail、schema hash 和 sidecar 写出。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'research.data_canonical_snapshot_v1',
 'status': 'completed',
 'snapshot_dir': 'research/data_canonical_snapshot_v1/canonical_snapshot',
 'row_count': 2,
 'dropped_non_calendar_rows': 1,
 'frame_schema_hash': 'd50304bef3df60b6ae09aa909f0226caf58f6f1b8a4d5b99e89f3074383613cb',
 'dataset_schema_hash': 'b4b81ddd4b6bf9bfe93432e51202b2eb77abb08809e3906c46d02a9693d73b02',
 'future_guardrail_error': 'future rows after as_of=2026-01-05: first=2026-01-06',
 'conclusion': 'canonical snapshot 已完成 qlib calendar 对齐、future row guardrail、schema hash 和 sidecar 写出。'}

## 中文结论

- 做了什么：新增并验证 `src.data` 的 canonical snapshot 构建和写出接口。
- 核心指标：保留 2 行 calendar 对齐数据，剔除 1 行非交易日数据，生成 64 位 `frame_schema_hash` 和 `dataset_schema_hash`。
- regime 差异：本轮是数据层能力，不做 regime 分析；它为后续 regime/window 分析提供一致的数据边界。
- 是否晋升：这是稳定 `src/data` 框架能力，可保留；不是策略晋升。
- 风险：示例写入 research card，真实数据刷新需要把 output_dir 指向受控的 `data/` 目录并纳入 schedule。
- 下一步：把真实 SOXL support panel 通过该接口写成 canonical snapshot，并接到 factor/model pipeline。